In [ ]:
import pandas as pd
import os
from tqdm import tqdm
from pydantic import BaseModel, model_validator
from typing import Optional
import ollama

# ingestion path
DATA_PATH = '../data/cleaned/cleaned_comments_info.csv'
LOG_PATH = '../data/cleaned/extract_log.txt'
OUT_PATH = '../data/cleaned/extracted_professions.csv'

# model prompt
PROMPT = """
Extraia se o canal pertence a um profissional de saúde humana.

Use title, description e keywords.

Prioridade de evidência:
1. title e description indicam declaração direta do autor/canal.
2. keywords podem ajudar, mas não bastam sozinhas para marcar true.
3. Se a profissão aparecer apenas como tema, curso, público-alvo, instituição, entrevistado, paciente ou estudante, retorne false.

Regras:
- true somente se o próprio canal/autoria se declarar profissional de SAÚDE HUMANA.
- Aceite profissão no title junto ao nome do canal, ex: "João Silva | Psicólogo".
- Use a profissão principal no masculino singular: Médico, Psicólogo, Psiquiatra, Nutricionista, Enfermeiro, Farmacêutico etc.
- Se houver especialidade médica, prefira a especialidade: Psiquiatra, Cardiologista, Dermatologista etc.
- Extraia registros como CRM, CRP, CRN, CREFITO, COREN, CRF, RQE, etc. Ex: "CRM-SP 123456", "CRP 06/12345"

Excluir:
- NÃO aceite profissão fora da saúde humana, como veterinário, professor, pastor, jornalista, advogado, coach, etc.

Retorne somente JSON válido neste schema:
{
  "declared_professional": true ou false,
  "profession": "Nome da Profissão" ou null,
  "license_number": "Registro" ou null
}
"""

In [ ]:
df_comments = pd.read_csv(DATA_PATH)
df_comments.head()

,channel_id,title,description,published_at,view_count,comment_count,subscriber_count,video_count,is_verified,keywords,profile_picture_url
0,UCGH4Wx-WL10iB4ZAdewkuDw,Adrenaline,O Adrenaline é um dos maiores e mais tradicion...,2005-12-09 21:58:36,307822353,0,1220000,4719,False,adrenaline,https://yt3.ggpht.com/SMveOOAq3F-AmS7pCeo4WgaD...
1,UCNgG7OI0MgvkxCEXVu10XRg,André Mod,Canal do ANDRÉ MOD - Curitiba - Paraná - Brasi...,2006-02-12 12:52:10,7355214,0,15600,402,False,Samba Rock Brasil Musica Brasileira Reggae Jam...,https://yt3.ggpht.com/ytc/AIdro_k6QrpTLmCrHvH9...
2,UC79nypvXFvfVGf-56PyyWFQ,Giselle Magalhães,"Meu nome é Giselle, sou paciente com Esclerose...",2006-02-14 21:45:46,28109,0,182,108,False,NaN,https://yt3.ggpht.com/ArxfqI_eWp-mANg32SCwY0IQ...
3,UC-PjJT0RVURj_L6D8D3WIww,Castro Brothers,Um bando de gente achando que tá fazendo algo ...,2006-02-24 13:11:06,1674441496,0,5520000,2418,False,comédia humor vlog piadas games marcos castro ...,https://yt3.ggpht.com/ytc/AIdro_n78kF_JG6jah5_...
4,UCmYuiV9wAR3i8p19Ygc7izg,Folha de S.Paulo,"Entrevistas, documentários, análises e reporta...",2006-03-01 13:16:04,305402733,0,1090000,10948,False,"""folha de são paulo"" folha ""folha de s.paulo"" ...",https://yt3.ggpht.com/hueB58GarYamE3VfoTU_kiHu...


In [3]:
# schema for ref
class ProfessionalInfo(BaseModel):
    declared_professional: Optional[bool] = False
    profession: Optional[str] = None
    license_number: Optional[str] = None

    @model_validator(mode='after')
    def fix_consistency(self):
        if self.profession is not None or self.license_number is not None:
            self.declared_professional = True
        elif self.profession is None and self.license_number is None:
            self.declared_professional = False
        return self

In [4]:
# functions for logging results
def read_log():
    """Reads the log file and returns a set of processed channel IDs."""
    if not os.path.exists(LOG_PATH):
        return set()
    with open(LOG_PATH, 'r') as l:
        channels = set(line.strip() for line in l if line.strip())
        return channels
    
def write_log(channel_id: str):
    """Appends a processed channel ID to the log file."""
    with open(LOG_PATH, 'a') as l:
        l.write(channel_id + '\n')

In [5]:
def call_ollama_model(channel_text):
    """Queries the local Ollama model with the channel text and returns the response string."""
    response = ollama.chat(
        model='gemma4:e2b',
        messages=[
            {'role': 'system', 'content': PROMPT},
            {'role': 'user', 'content': channel_text}
        ]
    )
    return response['message']['content']

In [6]:
def clean_text(value) -> str:
    """Converts empty/NaN values to an empty string."""
    if pd.isna(value):
        return ''
    return str(value).strip()


def classify(title: str, description: str, keywords: str) -> dict:
    """Classifies the profession from channel metadata and returns a validated dictionary."""
    title = clean_text(title)
    description = clean_text(description)
    keywords = clean_text(keywords)

    if not any([title, description, keywords]):
        return {"declared_professional": False, "profession": None, "license_number": None}

    try:
        channel_text = f"""title: {title}
                        description: {description}
                        keywords: {keywords}"""

        response_str = call_ollama_model(channel_text)

        if response_str:
            start_idx = response_str.find('{')
            end_idx = response_str.rfind('}')

            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                clean_json = response_str[start_idx:end_idx + 1]
                validated_data = ProfessionalInfo.model_validate_json(clean_json)
                return validated_data.model_dump()

        return {"declared_professional": False, "profession": None, "license_number": None}

    except Exception as e:
        raise

In [ ]:
list_results = []
processed_channels = read_log() 

for row in tqdm(df_comments['channel_id'].itertuples(), total=len(df_comments), desc="Processing..."):
    channel_id = str(row.channel_id)
    
    # checks if channel was already processed
    if channel_id in processed_channels:
        continue
    else:
        # classify each channel metadata and adds the identifier
        result = classify(row.title, row.description, row.keywords)
        result['channel_id'] = channel_id
        
        # saves csv
        list_results.append(result)
        df_temp = pd.DataFrame([result])
        
        # ensures correct column order before appending
        df_temp = df_temp[['channel_id', 'declared_professional', 'profession', 'license_number']]
        
        file_exists = os.path.exists(OUT_PATH)
        df_temp.to_csv(OUT_PATH, mode='a', header=not file_exists, index=False)
        
        # add the result to a pandas df and add log
        write_log(channel_id)

Processing...:   2%|▏         | 345/19086 [05:55<10:52:20,  2.09s/it]

In [ ]:
df_professions = pd.read_csv(OUT_PATH)

In [ ]:
# adding professional info to channels df
df_comments = df_comments.merge(
    df_professions[['channel_id', 'declared_professional']],
    on='channel_id',
    how='left'
)

# saving to csv
df_comments.to_csv(DATA_PATH, index=False, encoding='utf-8')